In [13]:
import pandas as pd
import torch
import sys
sys.path.append('../../')
from utilities import load_embedding

In [3]:
Crick_all = pd.read_csv('/mnt/zzbnew/peixunban/chenyihao/fluProfiler_source/src/Section1_modeling_antigenicity/1.5_Model_compare/data/all.csv')

In [4]:
Crick_all.columns

Index(['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d', 'seq_a', 'seq_b',
       'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat', 'serumName',
       'virusName', 'serumDate', 'virusDate', 'Type', 'serumIslID',
       'virusIslID', 'label', 'serumHA', 'virusHA', 'seq_diff_mat',
       'seq_diff_ohe'],
      dtype='object')

In [6]:
Crick_HA = Crick_all[['seq_id_c', 'seq_c', 'virusPassCat','virusName', 'virusDate', 'Type', 'virusIslID', 'label']].copy()

In [11]:
H1 = Crick_HA[Crick_HA['Type'] == 'H1N1'].sample(100)
H3 = Crick_HA[Crick_HA['Type'] == 'H3N2'].sample(100)

In [ ]:
device = torch.device('cuda:0')
embedding_df = pd.concat([H1, H3], axis=0)
sequence_names = pd.concat([embedding_df['seq_id_a'], embedding_df['seq_id_b'], 
                            embedding_df['seq_id_c'], embedding_df['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/mnt/zzbnew/peixunban/chenyihao/embedding", files=sequence_names)
embeddings = [emb.to(device) for emb in embeddings]
emb_dict = dict(zip(IDs, embeddings))

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3D 绘图支持

# ===== 1. 假设你已经有特征向量 =====
# 这里用随机数据模拟：100个样本，每个有50个特征
np.random.seed(42)
X = np.random.rand(100, 50)

# ===== 2. KMeans 聚类 =====
n_clusters = 4  # 假设分成 4 类
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
labels = kmeans.fit_predict(X)

# ===== 3. PCA 降维到 3D =====
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)

# ===== 4. 3D 绘图 =====
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')

# 不同簇不同颜色
for cluster_id in range(n_clusters):
    idx = labels == cluster_id
    ax.scatter(X_pca[idx, 0], X_pca[idx, 1], X_pca[idx, 2], label=f'Cluster {cluster_id}')

ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.set_zlabel('PC3')
ax.set_title('KMeans Clustering (PCA 3D Visualization)')
ax.legend()
plt.show()
